In [5]:

from pathlib import Path
import time
import pandas as pd
import numpy as np

start = time.perf_counter()

# Project paths
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

dataset_name = "maharashtra_agriculture_dataset.xlsx"
dataset_matches = [
    path
    for path in PROJECT_ROOT.rglob("*.xlsx")
    if path.name.lower() == dataset_name.lower()
]

DATA_PATH = (
    dataset_matches[0]
    if dataset_matches
    else PROJECT_ROOT / "data" / "raw" / dataset_name
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_PATH)

# Load dataset
# Resolve the dataset path safely and fail with a clear message
dataset_candidates = []
for base in [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT.parent,
    PROJECT_ROOT.parent / "data",
    PROJECT_ROOT.parent / "data" / "raw",
]:
    if base.exists():
        dataset_candidates.extend(
            p
            for p in base.rglob("*")
            if p.suffix.lower() in {".xlsx", ".xls"}
            and p.name.lower() == dataset_name.lower()
        )

if dataset_candidates:
    DATA_PATH = sorted(dataset_candidates, key=lambda p: len(p.parts))[0]
else:
    DATA_PATH = PROJECT_ROOT / "data" / "raw" / dataset_name

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}\n"
        "Check that the Excel file exists and that the filename matches "
        f"'{dataset_name}'.\n"
        f"Project root: {PROJECT_ROOT}"
    )

print("Loading Excel file...")

load_start = time.perf_counter()
df = pd.read_excel(DATA_PATH, engine="openpyxl")

print(f"Loaded in {time.perf_counter() - load_start:.2f} seconds")
print("Original shape:", df.shape)

# Standardize column names
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

# Remove duplicate rows
print("Checking duplicates...")
before = len(df)
df.drop_duplicates(inplace=True)

print("Duplicates removed:", before - len(df))
print("Final shape:", df.shape)

display(df.head())

print(f"Total execution time: {time.perf_counter() - start:.2f} seconds")


Project root: c:\Projects\AI-Driven Crop Yield Recommendation System\python
Dataset path: c:\Projects\AI-Driven Crop Yield Recommendation System\python\data\raw\maharashtra_agriculture_dataset.xlsx
Loading Excel file...
Loaded in 0.60 seconds
Original shape: (2500, 35)
Checking duplicates...
Duplicates removed: 0
Final shape: (2500, 35)


,record_id,district,city,latitude,longitude,soil_type,soil_color,soil_ph,nitrogen_kg_ha,phosphorus_kg_ha,...,production_kg,market_price_rs_kg,total_cost_rs,revenue_rs,profit_rs,drought_risk_pct,flood_risk_pct,overall_risk_pct,recommended_crop,recommendation_score
0,1,Aurangabad,Chhatrapati Sambhajinagar,19.88,75.34,Black,Black,7.82,262.9,8.3,...,23856.2,64.8,817651.4,1546569.3,728918.0,16.5,21.1,26.8,Gram,88.9
1,2,Sangli,Sangli,16.85,74.58,Black,Black,6.95,248.6,24.6,...,64257.4,101.6,964374.1,6530727.3,5566353.1,17.8,24.6,28.4,Tur,86.5
2,3,Gondia,Gondia,21.46,80.20,Alluvial,Dark Brown,6.68,233.8,25.5,...,1246675.0,12.9,2393831.0,16051010.4,13657179.4,19.9,27.6,31.6,Rice,82.9
3,4,Gadchiroli,Gadchiroli,20.18,79.99,Red Loamy,Red,5.85,117.8,21.5,...,11167.6,65.0,181452.6,726359.5,544907.0,0.0,46.5,35.6,Rice,77.3
4,5,Parbhani,Parbhani,19.27,76.78,Black,Black,6.81,183.7,15.0,...,5016.5,65.3,125770.1,327441.6,201671.5,0.0,32.1,25.3,Groundnut,81.1


Total execution time: 0.62 seconds


In [7]:

data = df.copy()

data = data.drop_duplicates().reset_index(drop=True)

text_columns = data.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    data[col] = data[col].astype("string").str.strip()
    data[col] = data[col].replace("", pd.NA)

print("Shape after removing duplicates:", data.shape)
display(data.isna().sum().sort_values(ascending=False).head(20))


Shape after removing duplicates: (2500, 35)


record_id                       0
district                        0
city                            0
latitude                        0
longitude                       0
soil_type                       0
soil_color                      0
soil_ph                         0
nitrogen_kg_ha                  0
phosphorus_kg_ha                0
potassium_kg_ha                 0
organic_carbon_pct              0
soil_moisture_pct               0
electrical_conductivity_ds_m    0
avg_temperature_c               0
rainfall_mm                     0
humidity_pct                    0
crop                            0
crop_season                     0
crop_duration_days              0
dtype: int64

In [8]:

numeric_candidates = [
    "Latitude",
    "Longitude",
    "pH",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "Organic_Carbon",
    "Moisture",
    "EC",
    "Temperature",
    "Rainfall",
    "Humidity",
    "Crop_Duration",
    "Farm_Area",
    "Fertilizer_Quantity",
    "Production",
    "Yield_per_Acre",
    "Cost",
    "Revenue",
    "Profit"
]

for col in numeric_candidates:
    if col in data.columns:
        data[col] = pd.to_numeric(
            data[col],
            errors="coerce"
        )

display(data.dtypes.to_frame("Data_Type"))


,Data_Type
record_id,int64
district,string
city,string
latitude,float64
longitude,float64
soil_type,string
soil_color,string
soil_ph,float64
nitrogen_kg_ha,float64
phosphorus_kg_ha,float64


In [9]:

# Combine soil nutrients when all three values exist.
if {
    "Nitrogen", "Phosphorus", "Potassium"
}.issubset(data.columns):

    data["NPK_Total"] = (
        data["Nitrogen"]
        + data["Phosphorus"]
        + data["Potassium"]
    )

# Categorize soil pH.
if "pH" in data.columns:

    data["Soil_pH_Category"] = pd.cut(
        data["pH"],
        bins=[
            -np.inf, 5.5, 6.5, 7.5, 8.5, np.inf
        ],
        labels=[
            "Strongly Acidic",
            "Moderately Acidic",
            "Near Neutral",
            "Alkaline",
            "Strongly Alkaline"
        ]
    )

# Rainfall per degree as a simple exploratory feature.
if {"Rainfall", "Temperature"}.issubset(data.columns):

    data["Rainfall_Temperature_Ratio"] = (
        data["Rainfall"]
        / data["Temperature"].replace(0, np.nan)
    )

print("New shape:", data.shape)
display(data.head())


New shape: (2500, 35)


,record_id,district,city,latitude,longitude,soil_type,soil_color,soil_ph,nitrogen_kg_ha,phosphorus_kg_ha,...,production_kg,market_price_rs_kg,total_cost_rs,revenue_rs,profit_rs,drought_risk_pct,flood_risk_pct,overall_risk_pct,recommended_crop,recommendation_score
0,1,Aurangabad,Chhatrapati Sambhajinagar,19.88,75.34,Black,Black,7.82,262.9,8.3,...,23856.2,64.8,817651.4,1546569.3,728918.0,16.5,21.1,26.8,Gram,88.9
1,2,Sangli,Sangli,16.85,74.58,Black,Black,6.95,248.6,24.6,...,64257.4,101.6,964374.1,6530727.3,5566353.1,17.8,24.6,28.4,Tur,86.5
2,3,Gondia,Gondia,21.46,80.20,Alluvial,Dark Brown,6.68,233.8,25.5,...,1246675.0,12.9,2393831.0,16051010.4,13657179.4,19.9,27.6,31.6,Rice,82.9
3,4,Gadchiroli,Gadchiroli,20.18,79.99,Red Loamy,Red,5.85,117.8,21.5,...,11167.6,65.0,181452.6,726359.5,544907.0,0.0,46.5,35.6,Rice,77.3
4,5,Parbhani,Parbhani,19.27,76.78,Black,Black,6.81,183.7,15.0,...,5016.5,65.3,125770.1,327441.6,201671.5,0.0,32.1,25.3,Groundnut,81.1


In [10]:

numeric_columns = data.select_dtypes(
    include=np.number
).columns.tolist()

categorical_columns = data.select_dtypes(
    exclude=np.number
).columns.tolist()

for col in numeric_columns:
    data[col] = data[col].fillna(
        data[col].median()
    )

for col in categorical_columns:
    mode_values = data[col].mode(dropna=True)

    if not mode_values.empty:
        data[col] = data[col].fillna(
            mode_values.iloc[0]
        )

print("Missing cells after imputation:", data.isna().sum().sum())


Missing cells after imputation: 0


In [11]:

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = (
    processed_dir
    / "maharashtra_agriculture_engineered.csv"
)

data.to_csv(output_path, index=False)

print("Exported:", output_path.resolve())
print("Final shape:", data.shape)


Exported: C:\Projects\AI-Driven Crop Yield Recommendation System\data\processed\maharashtra_agriculture_engineered.csv
Final shape: (2500, 35)


In [12]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Select the intended target for yield prediction.
target = "Yield_per_Acre"

if target in data.columns:

    # Remove target and direct outcome/leakage variables from predictors.
    leakage_columns = [
        "Production",
        "Revenue",
        "Profit",
        "Recommended_Crop",
        "Recommendation_Score"
    ]

    excluded = [
        col for col in leakage_columns
        if col in data.columns and col != target
    ]

    model_data = data.dropna(subset=[target]).copy()

    X = model_data.drop(
        columns=[target] + excluded,
        errors="ignore"
    )

    y = model_data[target]

    # Remove identifier columns where present.
    id_columns = [
        col for col in X.columns
        if col.lower().endswith("_id")
    ]

    X = X.drop(columns=id_columns, errors="ignore")

    numeric_features = X.select_dtypes(
        include=np.number
    ).columns.tolist()

    categorical_features = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    print("Target:", target)
    print("Rows available:", len(X))
    print("Numerical predictors:", numeric_features)
    print("Categorical predictors:", categorical_features)
    print("Excluded leakage columns:", excluded)

else:
    print(
        "Yield target not found. Update target to the "
        "correct column name before model preprocessing."
    )


Yield target not found. Update target to the correct column name before model preprocessing.
